# 07 - Pirate Agent with Tools

Run [06_prompt_agent.ipynb](06_prompt_agent.ipynb) first, or try the no-tools pirate agent in the Foundry playground. Use **Python (byom01-capability-testing)** with the working directory set to `notebooks/`.

This notebook uploads ten dummy planning documents about the town of Brackenford-on-Wye (in `data/brackenford/`) to a vector store and gives a pirate agent on the connected model **file search** and **code interpreter** in two ways:

- **Tool agent:** the tools are attached directly to the agent. This is the reliable option and is used for the questions below.
- **Toolbox agent:** the same tools are bundled in a toolbox and reached through its MCP endpoint. The toolbox is a preview feature and fails intermittently with `InternalServerError` (500), so the final section compares the two.

**Paid inference:** the questions make Foundry Responses requests through the gateway. Run only with approval.

The cleanup section deletes the vector store, toolbox, conversation and both agent versions created here. It leaves the connected model and gateway unchanged.


## Load Gateway Settings


In [ ]:
import json
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential

demo_root = Path("..").resolve()
gateway = json.loads((demo_root / "env" / "gateway.json").read_text(encoding="utf-8"))
credential = AzureCliCredential(subscription=gateway["subscriptionId"])
project_client = AIProjectClient(endpoint=gateway["projectEndpoint"], credential=credential)
openai_client = project_client.get_openai_client(max_retries=0)
print(f"Project: {gateway['projectId']}")
print(f"Connected model: {gateway['modelReference']}")


## Upload Documents to a Vector Store

Upload the ten Brackenford-on-Wye documents. File search reads from this vector store.

Uploading a file again does not overwrite the earlier copy; it adds a duplicate. To pick up edited documents, this cell deletes any existing `byom01-brackenford` vector store and recreates it. Re-run the tool agent, toolbox and toolbox agent cells afterwards so they point at the new store.


In [ ]:
docs = sorted((demo_root / "data" / "brackenford").glob("*.docx"))
for old in openai_client.vector_stores.list():
    if old.name == "byom01-brackenford":
        openai_client.vector_stores.delete(old.id)
vector_store = openai_client.vector_stores.create(name="byom01-brackenford")
batch = openai_client.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id,
    files=[path.open("rb") for path in docs],
)
print(f"Vector store: {vector_store.id}, files: {batch.file_counts.completed}/{len(docs)} completed")


## Create Tool Agent

Same pirate agent as notebook 06, with file search and code interpreter attached directly.


In [ ]:
from azure.ai.projects.models import (
    AutoCodeInterpreterToolParam,
    CodeInterpreterTool,
    FileSearchTool,
    MCPTool,
    PromptAgentDefinition,
)

PIRATE_INSTRUCTIONS = (
    "You are a helpful assistant. Always speak like a pirate. "
    "Use file search for questions about Brackenford-on-Wye and code interpreter for calculations."
)
tool_agent = project_client.agents.create_version(
    agent_name="byom01-pirate-tool-agent",
    definition=PromptAgentDefinition(
        model=gateway["modelReference"],
        instructions=PIRATE_INSTRUCTIONS,
        tools=[
            FileSearchTool(vector_store_ids=[vector_store.id]),
            CodeInterpreterTool(container=AutoCodeInterpreterToolParam()),
        ],
    ),
)
print(f"Tool agent: {tool_agent.name}, version: {tool_agent.version}")


## Create Toolbox

A toolbox bundles tools behind one managed endpoint. This one holds file search over the vector store and code interpreter.


In [ ]:
from azure.ai.projects.models import CodeInterpreterToolboxTool, FileSearchToolboxTool

toolbox = project_client.toolboxes.create_version(
    name="byom01-toolbox",
    description="File search over Brackenford-on-Wye documents plus code interpreter",
    tools=[
        FileSearchToolboxTool(
            name="brackenford_search",
            description="Search the Brackenford-on-Wye planning and council documents.",
            vector_store_ids=[vector_store.id],
        ),
        CodeInterpreterToolboxTool(
            name="calculator",
            description="Run Python for calculations, tables and charts.",
        ),
    ],
)
project_client.toolboxes.update(toolbox.name, default_version=toolbox.version)
print(f"Toolbox: {toolbox.name}, default version: {toolbox.version}")


In [ ]:
print("Default version:", project_client.toolboxes.get(toolbox.name).default_version)
for version in project_client.toolboxes.list_versions(toolbox.name):
    print("Version", version.version)
    for tool in version.tools:
        print("  ", tool.as_dict())


## Create Toolbox Agent

The same pirate agent, connected to the toolbox endpoint instead of holding the tools itself.


In [ ]:
toolbox_url = f"{gateway['projectEndpoint'].rstrip('/')}/toolboxes/{toolbox.name}/mcp?api-version=v1"
toolbox_agent = project_client.agents.create_version(
    agent_name="byom01-pirate-toolbox-agent",
    definition=PromptAgentDefinition(
        model=gateway["modelReference"],
        instructions=PIRATE_INSTRUCTIONS,
        tools=[MCPTool(server_label="toolbox", server_url=toolbox_url, require_approval="never")],
    ),
)
print(f"Toolbox agent: {toolbox_agent.name}, version: {toolbox_agent.version}")


## Ask a Question Answered by File Search

The answer comes from the uploaded documents. The output lists the tool calls the agent made. Questions go to the tool agent; set `active_agent = toolbox_agent` in the next cell to send them through the toolbox instead and expect occasional 500 errors.


In [ ]:
active_agent = tool_agent


In [ ]:
conversation = openai_client.conversations.create()


def ask(question):
    response = openai_client.responses.create(
        conversation=conversation.id,
        input=question,
        extra_body={"agent_reference": {"name": active_agent.name, "type": "agent_reference"}},
    )
    assert response.status == "completed", f"Foundry response did not complete: {response.status}"
    print(response.output_text)
    print("Tool calls:", [item.type for item in response.output if item.type != "message"])


ask("What infrastructure does the Local Plan Review say the housing allocation must include for healthcare, schools and a local centre?")


## Ask a Question Answered by Code Interpreter

This needs both tools: file search finds the site figures, then code interpreter calculates the total and percentages.


In [ ]:
ask(
    "The Local Plan Review lists the proposed housing sites and their homes. "
    "Find them, then use code to calculate the total number of homes and each site's percentage of that total."
)


## More Questions

Further questions that combine file search with code interpreter calculations and charts.


In [ ]:
ask("Using the pupils-per-home figures in the Local Plan Review, how many primary and secondary pupils will each housing site generate? Chart it.")


In [ ]:
ask("Using the cost per place, what is the total school contribution developers should pay, by phase of education?")


In [ ]:
ask("Compare the 2031 forecast surplus places with the pupils the new homes will generate. Is a two-form-entry primary school justified?")


In [ ]:
ask("Calculate homes per hectare and the number of affordable homes for each site. Which site is densest?")


In [ ]:
ask("From the Community Facilities report, calculate each school's yearly growth rate from 2022 to 2026 and project pupil numbers to 2031.")


In [ ]:
ask("Work out the cost per place for each school building project and flag any outliers.")


In [ ]:
ask("Chart how primary and secondary pupils travel to school. How far is the town from its 60% walk-and-cycle target?")


In [ ]:
ask("Check whether the secondary school expansion covers the pupils the housing will generate.")


## Compare the Tool Agent with the Toolbox Agent

Each cell sends a question that always needs file search 20 times to one agent and counts the failures.

In testing, the toolbox agent failed 7 of 20 calls with `InternalServerError` (500), while the tool agent and an agent with no tools failed none. Both agents use the same model, gateway, vector store and tools, so the toolbox endpoint is the likely source of the errors.


In [ ]:
RUNS = 20
QUESTION = "What is the total number of homes across the five housing sites in the Local Plan Review?"


def failure_rate(agent_name):
    failures = 0
    for i in range(RUNS):
        try:
            response = openai_client.responses.create(
                input=QUESTION,
                extra_body={"agent_reference": {"name": agent_name, "type": "agent_reference"}},
            )
            print(i + 1, response.status, [item.type for item in response.output if item.type != "message"])
        except Exception as error:
            failures += 1
            print(i + 1, type(error).__name__, getattr(error, "request_id", None))
    print(f"{agent_name}: {failures} of {RUNS} failed")


In [ ]:
failure_rate(toolbox_agent.name)


In [ ]:
failure_rate(tool_agent.name)


## Clean Up Test Resources

Deletes the conversation, both agent versions, toolbox and vector store created by this notebook. The connected model and gateway are retained.


In [ ]:
openai_client.conversations.delete(conversation_id=conversation.id)
project_client.agents.delete_version(
    agent_name=toolbox_agent.name, agent_version=toolbox_agent.version, force=True,
)
project_client.agents.delete_version(
    agent_name=tool_agent.name, agent_version=tool_agent.version, force=True,
)
project_client.toolboxes.delete(toolbox.name)
openai_client.vector_stores.delete(vector_store.id)
print("Conversation, agent versions, toolbox and vector store deleted; connection and gateway retained.")
